# Titanic Dataset — 20 Data Cleaning Prompts with Gemini API

This Colab notebook performs **exactly 20 distinct data-cleaning prompts** on `titanic.csv`.

Each prompt includes:
- a Markdown explanation of the cleaning objective,
- a reusable **Gemini API prompt**,
- executable Pandas cleaning code,
- a visible output showing the result.

The notebook defaults to `RUN_GEMINI = False` so it can be executed without an API key. Set it to `True` after adding your Gemini API key to get Gemini's live response for every prompt.

> **Dataset:** Titanic (`891` rows × `12` columns).


## 0. Setup and load the dataset

Google's current Python SDK is `google-genai`. The notebook uses `genai.Client()` and `client.models.generate_content(...)`.

**Security:** do not hard-code or publish your real API key.


In [1]:
# Install the current Gemini Python SDK in Colab.
print("Gemini SDK installation cell is included for Colab; skipped during local precomputation.")

import os
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

RUN_GEMINI = False  # Change to True after adding GEMINI_API_KEY.
GEMINI_MODEL = "gemini-3.8-flash"

def get_gemini_client():
    from google import genai
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY is not set.")
    return genai.Client(api_key=api_key)

def ask_gemini(prompt):
    if not RUN_GEMINI:
        return "Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live."
    client = get_gemini_client()
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return response.text

DATA_PATH = "titanic.csv"
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Please upload titanic.csv ...")
        uploaded = files.upload()
        if DATA_PATH not in uploaded:
            raise FileNotFoundError(f"{DATA_PATH} was not uploaded.")
    except ImportError:
        raise FileNotFoundError("Place titanic.csv in the working directory before running the notebook.")

df = pd.read_csv(DATA_PATH)
print("Loaded dataset:", df.shape)
display(df.head())


Gemini SDK installation cell is included for Colab; skipped during local precomputation.
Loaded dataset: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 01. Audit duplicate records

**What this prompt cleans:** Identify and handle duplicate rows in the Titanic dataset. Report the number of duplicated rows before cleaning, remove exact duplicate records while keeping the first occurrence, and verify that no duplicates remain.

**Gemini prompt:**  
> Identify and handle duplicate rows in the Titanic dataset. Report the number of duplicated rows before cleaning, remove exact duplicate records while keeping the first occurrence, and verify that no duplicates remain.


In [2]:
PROMPT_01 = """Identify and handle duplicate rows in the Titanic dataset. Report the number of duplicated rows before cleaning, remove exact duplicate records while keeping the first occurrence, and verify that no duplicates remain."""
# Prompt 01: Audit and remove exact duplicate records
gemini_prompt = PROMPT_01
print(ask_gemini(gemini_prompt))

before = len(df)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Rows before: {before}")
print(f"Duplicate rows found: {dup_count}")
print(f"Rows after: {after}")
print(f"Remaining duplicate rows: {int(df.duplicated().sum())}")


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Rows before: 891
Duplicate rows found: 0
Rows after: 891
Remaining duplicate rows: 0


## 02. Standardize column names

**What this prompt cleans:** Standardize all Titanic column names into clean snake_case format: lowercase, spaces/special characters replaced with underscores, and duplicate column names made unique. Display the before/after column names.

**Gemini prompt:**  
> Standardize all Titanic column names into clean snake_case format: lowercase, spaces/special characters replaced with underscores, and duplicate column names made unique. Display the before/after column names.


In [3]:
PROMPT_02 = """Standardize all Titanic column names into clean snake_case format: lowercase, spaces/special characters replaced with underscores, and duplicate column names made unique. Display the before/after column names."""
# Prompt 02: Standardize column names
gemini_prompt = PROMPT_02
print(ask_gemini(gemini_prompt))

before_cols = df.columns.tolist()
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_")
)
after_cols = df.columns.tolist()

print("Before:", before_cols)
print("After :", after_cols)


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Before: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
After : ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


## 03. Audit and fill missing Age

**What this prompt cleans:** Clean the Age column by converting it to numeric, identifying missing values, and imputing missing ages using the median age calculated within Sex and Pclass groups. Report missing counts before and after.

**Gemini prompt:**  
> Clean the Age column by converting it to numeric, identifying missing values, and imputing missing ages using the median age calculated within Sex and Pclass groups. Report missing counts before and after.


In [4]:
PROMPT_03 = """Clean the Age column by converting it to numeric, identifying missing values, and imputing missing ages using the median age calculated within Sex and Pclass groups. Report missing counts before and after."""
# Prompt 03: Impute Age by Sex and Pclass group median
gemini_prompt = PROMPT_03
print(ask_gemini(gemini_prompt))

df["age"] = pd.to_numeric(df["age"], errors="coerce")
missing_before = int(df["age"].isna().sum())

group_medians = df.groupby(["sex", "pclass"])["age"].transform("median")
df["age"] = df["age"].fillna(group_medians)
df["age"] = df["age"].fillna(df["age"].median())

missing_after = int(df["age"].isna().sum())

print(f"Missing Age before: {missing_before}")
print(f"Missing Age after : {missing_after}")
display(df[["age", "sex", "pclass"]].head(10))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Missing Age before: 177
Missing Age after : 0


,age,sex,pclass
0,22.0,male,3
1,38.0,female,1
2,26.0,female,3
3,35.0,female,1
4,35.0,male,3
5,25.0,male,3
6,54.0,male,1
7,2.0,male,3
8,27.0,female,3
9,14.0,female,2


## 04. Clean missing Embarked values

**What this prompt cleans:** Clean the Embarked column by trimming whitespace, standardizing text case, treating blank strings as missing, and filling missing values with the overall mode. Show the counts before and after.

**Gemini prompt:**  
> Clean the Embarked column by trimming whitespace, standardizing text case, treating blank strings as missing, and filling missing values with the overall mode. Show the counts before and after.


In [5]:
PROMPT_04 = """Clean the Embarked column by trimming whitespace, standardizing text case, treating blank strings as missing, and filling missing values with the overall mode. Show the counts before and after."""
# Prompt 04: Clean and impute Embarked
gemini_prompt = PROMPT_04
print(ask_gemini(gemini_prompt))

df["embarked"] = df["embarked"].astype("string").str.strip().str.upper()
df["embarked"] = df["embarked"].replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
missing_before = int(df["embarked"].isna().sum())
mode_embarked = df["embarked"].mode(dropna=True).iloc[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)
missing_after = int(df["embarked"].isna().sum())

print(f"Missing Embarked before: {missing_before}")
print(f"Mode used for imputation: {mode_embarked}")
print(f"Missing Embarked after : {missing_after}")
print(df["embarked"].value_counts(dropna=False).to_string())


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Missing Embarked before: 2
Mode used for imputation: S
Missing Embarked after : 0
embarked
S    646
C    168
Q     77


## 05. Clean and flag Cabin missingness

**What this prompt cleans:** Treat missing Cabin values as a meaningful missingness condition rather than dropping rows. Create a binary `cabin_known` indicator and replace missing Cabin values with the label `Unknown`. Report the before/after missingness.

**Gemini prompt:**  
> Treat missing Cabin values as a meaningful missingness condition rather than dropping rows. Create a binary `cabin_known` indicator and replace missing Cabin values with the label `Unknown`. Report the before/after missingness.


In [6]:
PROMPT_05 = """Treat missing Cabin values as a meaningful missingness condition rather than dropping rows. Create a binary `cabin_known` indicator and replace missing Cabin values with the label `Unknown`. Report the before/after missingness."""
# Prompt 05: Preserve Cabin missingness with an indicator
gemini_prompt = PROMPT_05
print(ask_gemini(gemini_prompt))

df["cabin"] = df["cabin"].astype("string").str.strip()
missing_before = int(df["cabin"].isna().sum())
df["cabin_known"] = df["cabin"].notna().astype("int8")
df["cabin"] = df["cabin"].fillna("Unknown")
missing_after = int(df["cabin"].isna().sum())

print(f"Missing Cabin before: {missing_before}")
print(f"Missing Cabin after : {missing_after}")
print("cabin_known counts:")
print(df["cabin_known"].value_counts().sort_index().to_string())
display(df[["cabin", "cabin_known"]].head(8))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Missing Cabin before: 687
Missing Cabin after : 0
cabin_known counts:
cabin_known
0    687
1    204


,cabin,cabin_known
0,Unknown,0
1,C85,1
2,Unknown,0
3,C123,1
4,Unknown,0
5,Unknown,0
6,E46,1
7,Unknown,0


## 06. Normalize categorical text

**What this prompt cleans:** Normalize text-based categorical fields such as Sex and Embarked by trimming whitespace, converting to consistent case, and mapping obvious spelling/case variants to canonical values.

**Gemini prompt:**  
> Normalize text-based categorical fields such as Sex and Embarked by trimming whitespace, converting to consistent case, and mapping obvious spelling/case variants to canonical values.


In [7]:
PROMPT_06 = """Normalize text-based categorical fields such as Sex and Embarked by trimming whitespace, converting to consistent case, and mapping obvious spelling/case variants to canonical values."""
# Prompt 06: Normalize categorical text values
gemini_prompt = PROMPT_06
print(ask_gemini(gemini_prompt))

# Canonicalize Sex and Embarked.
df["sex"] = df["sex"].astype("string").str.strip().str.lower()
df["sex"] = df["sex"].replace({
    "m": "male", "man": "male", "male ": "male",
    "f": "female", "woman": "female", "female ": "female"
})
df["embarked"] = df["embarked"].astype("string").str.strip().str.upper()

print("Sex unique values:", sorted(df["sex"].dropna().unique().tolist()))
print("Embarked unique values:", sorted(df["embarked"].dropna().unique().tolist()))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Sex unique values: ['female', 'male']
Embarked unique values: ['C', 'Q', 'S']


## 07. Detect invalid categorical values

**What this prompt cleans:** Validate the categorical domains for Sex and Embarked. Identify any values outside the expected sets (`male/female` and `S/C/Q`) and report the invalid records without silently deleting them.

**Gemini prompt:**  
> Validate the categorical domains for Sex and Embarked. Identify any values outside the expected sets (`male/female` and `S/C/Q`) and report the invalid records without silently deleting them.


In [8]:
PROMPT_07 = """Validate the categorical domains for Sex and Embarked. Identify any values outside the expected sets (`male/female` and `S/C/Q`) and report the invalid records without silently deleting them."""
# Prompt 07: Detect invalid categorical values
gemini_prompt = PROMPT_07
print(ask_gemini(gemini_prompt))

valid_sex = {"male", "female"}
valid_embarked = {"S", "C", "Q"}

invalid_sex = df.loc[~df["sex"].isin(valid_sex), ["sex"]].drop_duplicates()
invalid_embarked = df.loc[~df["embarked"].isin(valid_embarked), ["embarked"]].drop_duplicates()

print("Invalid Sex values:")
display(invalid_sex if not invalid_sex.empty else pd.DataFrame({"result": ["None found"]}))

print("Invalid Embarked values:")
display(invalid_embarked if not invalid_embarked.empty else pd.DataFrame({"result": ["None found"]}))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Invalid Sex values:


,result
0,None found


Invalid Embarked values:


,result
0,None found


## 08. Coerce numeric data types

**What this prompt cleans:** Ensure numeric Titanic fields are stored as appropriate numeric types. Convert PassengerId, Survived, Pclass, SibSp, Parch, Age, and Fare to numeric, then display the resulting data types and any conversion-created missing values.

**Gemini prompt:**  
> Ensure numeric Titanic fields are stored as appropriate numeric types. Convert PassengerId, Survived, Pclass, SibSp, Parch, Age, and Fare to numeric, then display the resulting data types and any conversion-created missing values.


In [9]:
PROMPT_08 = """Ensure numeric Titanic fields are stored as appropriate numeric types. Convert PassengerId, Survived, Pclass, SibSp, Parch, Age, and Fare to numeric, then display the resulting data types and any conversion-created missing values."""
# Prompt 08: Coerce numeric columns
gemini_prompt = PROMPT_08
print(ask_gemini(gemini_prompt))

numeric_cols = ["passengerid", "survived", "pclass", "sibsp", "parch", "age", "fare"]
before_missing = df[numeric_cols].isna().sum()

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

after_missing = df[numeric_cols].isna().sum()

print("Dtypes after conversion:")
print(df[numeric_cols].dtypes.to_string())
print("\nMissing-value changes caused by numeric coercion:")
print((after_missing - before_missing).to_string())


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Dtypes after conversion:
passengerid      int64
survived         int64
pclass           int64
sibsp            int64
parch            int64
age            float64
fare           float64

Missing-value changes caused by numeric coercion:
passengerid    0
survived       0
pclass         0
sibsp          0
parch          0
age            0
fare           0


## 09. Validate PassengerId uniqueness

**What this prompt cleans:** Check the PassengerId field for missing values, duplicates, and uniqueness. Treat PassengerId as an identifier rather than a predictive feature and report any problems found.

**Gemini prompt:**  
> Check the PassengerId field for missing values, duplicates, and uniqueness. Treat PassengerId as an identifier rather than a predictive feature and report any problems found.


In [10]:
PROMPT_09 = """Check the PassengerId field for missing values, duplicates, and uniqueness. Treat PassengerId as an identifier rather than a predictive feature and report any problems found."""
# Prompt 09: Validate the identifier column
gemini_prompt = PROMPT_09
print(ask_gemini(gemini_prompt))

missing_ids = int(df["passengerid"].isna().sum())
duplicate_ids = int(df["passengerid"].duplicated().sum())
unique_ids = int(df["passengerid"].nunique(dropna=True))

print(f"Missing PassengerId: {missing_ids}")
print(f"Duplicate PassengerId values: {duplicate_ids}")
print(f"Unique PassengerId values: {unique_ids}")
print(f"Row count: {len(df)}")
print("Identifier status:", "PASS" if missing_ids == 0 and duplicate_ids == 0 and unique_ids == len(df) else "REVIEW")


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Missing PassengerId: 0
Duplicate PassengerId values: 0
Unique PassengerId values: 891
Row count: 891
Identifier status: PASS


## 10. Clean whitespace in Ticket values

**What this prompt cleans:** Clean Ticket strings by trimming leading/trailing whitespace and collapsing repeated internal whitespace. Preserve the ticket meaning while removing formatting noise.

**Gemini prompt:**  
> Clean Ticket strings by trimming leading/trailing whitespace and collapsing repeated internal whitespace. Preserve the ticket meaning while removing formatting noise.


In [11]:
PROMPT_10 = """Clean Ticket strings by trimming leading/trailing whitespace and collapsing repeated internal whitespace. Preserve the ticket meaning while removing formatting noise."""
# Prompt 10: Normalize Ticket strings
gemini_prompt = PROMPT_10
print(ask_gemini(gemini_prompt))

df["ticket"] = (
    df["ticket"].astype("string")
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
)

print("Sample cleaned Ticket values:")
display(df[["ticket"]].head(12))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Sample cleaned Ticket values:


,ticket
0,A/5 21171
1,PC 17599
2,STON/O2. 3101282
3,113803
4,373450
5,330877
6,17463
7,349909
8,347742
9,237736


## 11. Clean Name strings

**What this prompt cleans:** Clean passenger Name values by removing extra whitespace, collapsing repeated spaces, and standardizing surrounding punctuation without removing meaningful name information.

**Gemini prompt:**  
> Clean passenger Name values by removing extra whitespace, collapsing repeated spaces, and standardizing surrounding punctuation without removing meaningful name information.


In [12]:
PROMPT_11 = """Clean passenger Name values by removing extra whitespace, collapsing repeated spaces, and standardizing surrounding punctuation without removing meaningful name information."""
# Prompt 11: Clean Name strings
gemini_prompt = PROMPT_11
print(ask_gemini(gemini_prompt))

df["name"] = (
    df["name"].astype("string")
              .str.strip()
              .str.replace(r"\s+", " ", regex=True)
)

print("Sample cleaned Name values:")
display(df[["name"]].head(10))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Sample cleaned Name values:


,name
0,"Braund, Mr. Owen Harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,"Heikkinen, Miss. Laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,"Allen, Mr. William Henry"
5,"Moran, Mr. James"
6,"McCarthy, Mr. Timothy J"
7,"Palsson, Master. Gosta Leonard"
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)"
9,"Nasser, Mrs. Nicholas (Adele Achem)"


## 12. Extract and standardize honorific titles

**What this prompt cleans:** Extract the title from each passenger's Name (for example, Mr, Miss, Mrs, Master). Strip punctuation, group rare titles into `Rare`, and display the title frequency distribution.

**Gemini prompt:**  
> Extract the title from each passenger's Name (for example, Mr, Miss, Mrs, Master). Strip punctuation, group rare titles into `Rare`, and display the title frequency distribution.


In [13]:
PROMPT_12 = """Extract the title from each passenger's Name (for example, Mr, Miss, Mrs, Master). Strip punctuation, group rare titles into `Rare`, and display the title frequency distribution."""
# Prompt 12: Extract a clean title feature
gemini_prompt = PROMPT_12
print(ask_gemini(gemini_prompt))

df["title"] = df["name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip().str.title()

common_titles = {"Mr", "Miss", "Mrs", "Master"}
df.loc[~df["title"].isin(common_titles), "title"] = "Rare"

print("Title distribution:")
print(df["title"].value_counts(dropna=False).to_string())


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Title distribution:
title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27


## 13. Validate Age range

**What this prompt cleans:** Check Age for impossible or suspicious values. Treat negative ages and implausibly large ages as invalid, replace invalid ages with missing, then impute them using the overall median. Report the number of invalid values corrected.

**Gemini prompt:**  
> Check Age for impossible or suspicious values. Treat negative ages and implausibly large ages as invalid, replace invalid ages with missing, then impute them using the overall median. Report the number of invalid values corrected.


In [14]:
PROMPT_13 = """Check Age for impossible or suspicious values. Treat negative ages and implausibly large ages as invalid, replace invalid ages with missing, then impute them using the overall median. Report the number of invalid values corrected."""
# Prompt 13: Validate and repair Age range
gemini_prompt = PROMPT_13
print(ask_gemini(gemini_prompt))

age = pd.to_numeric(df["age"], errors="coerce")
invalid_age_mask = (age < 0) | (age > 100)
invalid_count = int(invalid_age_mask.sum())

df.loc[invalid_age_mask, "age"] = np.nan
median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)

print(f"Invalid Age values corrected: {invalid_count}")
print(f"Median used for any repaired Age values: {median_age:.2f}")
print(f"Age minimum after cleaning: {df['age'].min():.2f}")
print(f"Age maximum after cleaning: {df['age'].max():.2f}")


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Invalid Age values corrected: 0
Median used for any repaired Age values: 26.00
Age minimum after cleaning: 0.42
Age maximum after cleaning: 80.00


## 14. Detect Fare outliers with IQR

**What this prompt cleans:** Use the IQR rule to identify potential Fare outliers. Do not delete valid passengers just because a fare is unusually high; create an `fare_outlier` flag and summarize the flagged records.

**Gemini prompt:**  
> Use the IQR rule to identify potential Fare outliers. Do not delete valid passengers just because a fare is unusually high; create an `fare_outlier` flag and summarize the flagged records.


In [15]:
PROMPT_14 = """Use the IQR rule to identify potential Fare outliers. Do not delete valid passengers just because a fare is unusually high; create an `fare_outlier` flag and summarize the flagged records."""
# Prompt 14: Flag, don't blindly delete, Fare outliers
gemini_prompt = PROMPT_14
print(ask_gemini(gemini_prompt))

q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df["fare_outlier"] = ((df["fare"] < lower) | (df["fare"] > upper)).astype("int8")

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Lower bound: {lower:.2f}")
print(f"Upper bound: {upper:.2f}")
print(f"Potential Fare outliers flagged: {int(df['fare_outlier'].sum())}")

display(df.loc[df["fare_outlier"] == 1, ["passengerid", "pclass", "fare", "fare_outlier"]].head(10))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Q1: 7.91
Q3: 31.00
IQR: 23.09
Lower bound: -26.72
Upper bound: 65.63
Potential Fare outliers flagged: 116


,passengerid,pclass,fare,fare_outlier
1,2,1,71.2833,1
27,28,1,263.0000,1
31,32,1,146.5208,1
34,35,1,82.1708,1
52,53,1,76.7292,1
61,62,1,80.0000,1
62,63,1,83.4750,1
72,73,2,73.5000,1
88,89,1,263.0000,1
102,103,1,77.2875,1


## 15. Validate non-negative count variables

**What this prompt cleans:** Check SibSp and Parch for impossible negative values. Convert negatives to missing and impute them with zero because these columns represent counts of relatives aboard.

**Gemini prompt:**  
> Check SibSp and Parch for impossible negative values. Convert negatives to missing and impute them with zero because these columns represent counts of relatives aboard.


In [16]:
PROMPT_15 = """Check SibSp and Parch for impossible negative values. Convert negatives to missing and impute them with zero because these columns represent counts of relatives aboard."""
# Prompt 15: Validate count variables
gemini_prompt = PROMPT_15
print(ask_gemini(gemini_prompt))

count_cols = ["sibsp", "parch"]
invalid_counts = {}
for col in count_cols:
    mask = df[col] < 0
    invalid_counts[col] = int(mask.sum())
    df.loc[mask, col] = np.nan
    df[col] = df[col].fillna(0).astype("int64")

print("Invalid negative counts corrected:", invalid_counts)
print("Minimum values after cleaning:")
print(df[count_cols].min().to_string())


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Invalid negative counts corrected: {'sibsp': 0, 'parch': 0}
Minimum values after cleaning:
sibsp    0
parch    0


## 16. Standardize Survived and Pclass domains

**What this prompt cleans:** Validate binary and ordinal categorical variables: Survived must be 0/1 and Pclass must be 1/2/3. Report invalid values and cast them to nullable integer types for clean storage.

**Gemini prompt:**  
> Validate binary and ordinal categorical variables: Survived must be 0/1 and Pclass must be 1/2/3. Report invalid values and cast them to nullable integer types for clean storage.


In [17]:
PROMPT_16 = """Validate binary and ordinal categorical variables: Survived must be 0/1 and Pclass must be 1/2/3. Report invalid values and cast them to nullable integer types for clean storage."""
# Prompt 16: Validate Survived and Pclass domains
gemini_prompt = PROMPT_16
print(ask_gemini(gemini_prompt))

df["survived"] = pd.to_numeric(df["survived"], errors="coerce")
df["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")

invalid_survived = int((~df["survived"].isin([0, 1])).sum())
invalid_pclass = int((~df["pclass"].isin([1, 2, 3])).sum())

print(f"Invalid Survived values: {invalid_survived}")
print(f"Invalid Pclass values: {invalid_pclass}")

df["survived"] = df["survived"].astype("Int64")
df["pclass"] = df["pclass"].astype("Int64")

print("\nSurvived distribution:")
print(df["survived"].value_counts(dropna=False).sort_index().to_string())
print("\nPclass distribution:")
print(df["pclass"].value_counts(dropna=False).sort_index().to_string())


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Invalid Survived values: 0
Invalid Pclass values: 0

Survived distribution:
survived
0    549
1    342

Pclass distribution:
pclass
1    216
2    184
3    491


## 17. Create family_size

**What this prompt cleans:** Create a reliable family-size feature from SibSp and Parch using `family_size = SibSp + Parch + 1`. Use the cleaned numeric count fields and show a sample of the result.

**Gemini prompt:**  
> Create a reliable family-size feature from SibSp and Parch using `family_size = SibSp + Parch + 1`. Use the cleaned numeric count fields and show a sample of the result.


In [18]:
PROMPT_17 = """Create a reliable family-size feature from SibSp and Parch using `family_size = SibSp + Parch + 1`. Use the cleaned numeric count fields and show a sample of the result."""
# Prompt 17: Create family size from cleaned counts
gemini_prompt = PROMPT_17
print(ask_gemini(gemini_prompt))

df["family_size"] = df["sibsp"] + df["parch"] + 1

print("Family-size summary:")
display(df["family_size"].describe().to_frame().T)
display(df[["sibsp", "parch", "family_size"]].head(10))


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
Family-size summary:


,count,mean,std,min,25%,50%,75%,max
family_size,891.0,1.904602,1.613459,1.0,1.0,1.0,2.0,11.0


,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1
5,0,0,1
6,0,0,1
7,3,1,5
8,0,2,3
9,1,0,2


## 18. Create family_size_group

**What this prompt cleans:** Convert the numeric Family Size feature into readable groups: `Alone` for 1, `Small` for 2–4, `Medium` for 5–7, and `Large` for 8 or more. Display the frequency table.

**Gemini prompt:**  
> Convert the numeric Family Size feature into readable groups: `Alone` for 1, `Small` for 2–4, `Medium` for 5–7, and `Large` for 8 or more. Display the frequency table.


In [19]:
PROMPT_18 = """Convert the numeric Family Size feature into readable groups: `Alone` for 1, `Small` for 2–4, `Medium` for 5–7, and `Large` for 8 or more. Display the frequency table."""
# Prompt 18: Categorize family size
gemini_prompt = PROMPT_18
print(ask_gemini(gemini_prompt))

def family_group(n):
    if n == 1:
        return "Alone"
    elif 2 <= n <= 4:
        return "Small"
    elif 5 <= n <= 7:
        return "Medium"
    return "Large"

df["family_size_group"] = df["family_size"].apply(family_group)

print(df["family_size_group"].value_counts().to_string())


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.
family_size_group
Alone     537
Small     292
Medium     49
Large      13


## 19. Remove constant / all-missing columns

**What this prompt cleans:** Profile the cleaned dataset for columns that are entirely missing or contain only one unique value. Remove only columns that are all-missing or truly constant, and report what was removed.

**Gemini prompt:**  
> Profile the cleaned dataset for columns that are entirely missing or contain only one unique value. Remove only columns that are all-missing or truly constant, and report what was removed.


In [20]:
PROMPT_19 = """Profile the cleaned dataset for columns that are entirely missing or contain only one unique value. Remove only columns that are all-missing or truly constant, and report what was removed."""
# Prompt 19: Remove unusable constant columns
gemini_prompt = PROMPT_19
print(ask_gemini(gemini_prompt))

all_missing = [c for c in df.columns if df[c].isna().all()]
constant = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
to_drop = sorted(set(all_missing + constant))

df = df.drop(columns=to_drop)

print("All-missing columns:", all_missing if all_missing else "None")
print("Constant columns:", constant if constant else "None")
print("Dropped columns:", to_drop if to_drop else "None")
print("Remaining shape:", df.shape)


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.


All-missing columns: None
Constant columns: None
Dropped columns: None
Remaining shape: (891, 17)


## 20. Final data-quality validation

**What this prompt cleans:** Perform a final data-quality check after cleaning. Report remaining missing values, duplicate rows, duplicate PassengerId values, data types, and the final shape. Confirm whether the cleaned dataset passes the validation checks.

**Gemini prompt:**  
> Perform a final data-quality check after cleaning. Report remaining missing values, duplicate rows, duplicate PassengerId values, data types, and the final shape. Confirm whether the cleaned dataset passes the validation checks.


In [21]:
PROMPT_20 = """Perform a final data-quality check after cleaning. Report remaining missing values, duplicate rows, duplicate PassengerId values, data types, and the final shape. Confirm whether the cleaned dataset passes the validation checks."""
# Prompt 20: Final quality validation
gemini_prompt = PROMPT_20
print(ask_gemini(gemini_prompt))

missing_summary = df.isna().sum()
remaining_missing = int(missing_summary.sum())
duplicate_rows = int(df.duplicated().sum())
duplicate_ids = int(df["passengerid"].duplicated().sum())

print("Final shape:", df.shape)
print("Total remaining missing cells:", remaining_missing)
print("Remaining duplicate rows:", duplicate_rows)
print("Duplicate PassengerId values:", duplicate_ids)

print("\nRemaining missing values by column:")
print(missing_summary[missing_summary > 0].to_string() if (missing_summary > 0).any() else "None")

print("\nFinal data types:")
print(df.dtypes.to_string())

passed = (
    remaining_missing == 0
    and duplicate_rows == 0
    and duplicate_ids == 0
)
print("\nFINAL VALIDATION:", "PASS" if passed else "REVIEW REQUIRED")


Gemini execution is OFF. Set RUN_GEMINI=True to run this prompt live.


Final shape: (891, 17)
Total remaining missing cells: 0
Remaining duplicate rows: 0
Duplicate PassengerId values: 0

Remaining missing values by column:
None

Final data types:
passengerid                   int64
survived                      Int64
pclass                        Int64
name                 string[python]
sex                  string[python]
age                         float64
sibsp                         int64
parch                         int64
ticket               string[python]
fare                        float64
cabin                string[python]
embarked             string[python]
cabin_known                    int8
title                string[python]
fare_outlier                   int8
family_size                   int64
family_size_group            object

FINAL VALIDATION: PASS


## Final save

This saves the cleaned dataset after all 20 prompts.


In [22]:
OUTPUT_PATH = "titanic_cleaned_final.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
print(f"Final rows: {len(df)}")
print(f"Final columns: {len(df.columns)}")
display(df.head())


Saved cleaned dataset to: /mnt/data/titanic_cleaned_final.csv
Final rows: 891
Final columns: 17


,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,cabin_known,title,fare_outlier,family_size,family_size_group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Unknown,S,0,Mr,0,2,Small
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1,Mrs,1,2,Small
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,S,0,Miss,0,1,Alone
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1,Mrs,0,2,Small
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Unknown,S,0,Mr,0,1,Alone
